# Find the best encoding settings

The aim here is to train a model on existing data in order to estimate the FFmpeg parameters that optimise any arbitrary criterion.

In [1]:
import cutcutcodec

from mendevi.models import Solver
from mendevi.models.nn import EnergyNN, QualityNN

In [2]:
dataset = "scatter.db"  # the dataset name, automatically downloaded
val = 0.2  # keep 80% of data for training, 20% for validation
video = cutcutcodec.utils.get_project_root() / "media" / "video" / "intro.webm"

## Train the model

Before making predictions, you first need to train a model.

### Quality model

In [3]:
# declaration
quality_model = QualityNN(
    output_labels=["log_rate", "psnr", "ssim"],
    weights="quality",  # optional, for checkpoint (in ~/.cache/mendevi)
    niter=1000,  # number of training iterations, default 10000
    learning_rate=1e-5,  # adam gradient decay learning rate, default 1e-4
    total=1000,  # number of model parameters, default based on number of observations
)

In [4]:
# train
if not quality_model.is_fit():
    quality_model.fit(
        dataset,
        table="t_enc_encode",  # optional, help to find the good SQL request
        select=f"uniform(vid_hash) >= {val}",
    )

Fit 'Quality estimation based on a neuronal network'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster ...
█ █ this cluster contains 28562 observations
█ █ Save model into /home/rrichard/.cache/mendevi/quality_29_3_1000.pk.xz...                                         
█ █ █ 47.44 ms
█ █ 3 min 10 s
█ 8 min 56 s
Save model into /home/rrichard/.cache/mendevi/quality.pk.xz...
█ 123.04 ms


In [5]:
# validation
quality_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val}")

Validate 'Quality estimation based on a neuronal network'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster ...
█ █ this cluster contains 7215 observations
█ █ 1.06 s
█ 2 min 0 s
MAPE of log_rate: 9.73%
RMS  of log_rate: 0.7602
MAPE of psnr: 17.93%
RMS  of psnr: 8.467
MAPE of ssim: 7.96%
RMS  of ssim: 0.08902
MAPE of rate: 362.64%
RMS  of rate: 6.07e+07


### Energy model

In [6]:
energy_model = EnergyNN(
    weights="energy",  # optional, for checkpoint (in ~/.cache/mendevi)
    niter=1000,  # number of training iterations, default 10000
    learning_rate=1e-5,  # adam gradient decay learning rate, default 1e-4
    total=1000,  # number of model parameters, default based on number of observations
)

In [7]:
if not energy_model.is_fit():
    energy_model.fit(
        dataset,
        table="t_enc_encode",  # optional, help to find the good SQL request
        select=f"uniform(vid_hash) >= {val}",
    )

Fit 'Encoding energy estimation based on a neuronal network'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster hostname=abacus26-1.rennes.grid5000.fr...
█ █ this cluster contains 16164 observations
█ █ Save model into /home/rrichard/.cache/mendevi/energy_29_2_1000.pk.xz...                                          
█ █ █ 87.91 ms
█ █ 2 min 44 s
█ Fit on cluster hostname=dahu-12.grenoble.grid5000.fr...
█ █ this cluster contains 2539 observations
█ █ Load model from /home/rrichard/.cache/mendevi/energy_29_2_1000.pk.xz...
█ █ █ 17.31 ms
█ █ Save model into /home/rrichard/.cache/mendevi/energy_29_2_1000.pk.xz...                                          
█ █ █ 31.07 ms
█ █ 1 min 23 s
█ Fit on cluster hostname=gemini-2.lyon.grid5000.fr...
█ █ this cluster contains 1663 observations
█ █ Load model from /home/rrichard/.cache/mendevi/energy_29_2_1000.pk.xz...
█ █ █ 38.46 ms
█ █ Save model into /home/rrichard/.cache/mendevi/energy_29_2_1000.pk.xz...                                      

In [8]:
# validation
energy_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val}")

Validate 'Encoding energy estimation based on a neuronal network'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster hostname=abacus26-1.rennes.grid5000.fr...
█ █ this cluster contains 4068 observations
█ █ 752.04 ms
█ Validate on cluster hostname=dahu-12.grenoble.grid5000.fr...
█ █ this cluster contains 676 observations
█ █ 207.35 ms
█ Validate on cluster hostname=gemini-2.lyon.grid5000.fr...
█ █ this cluster contains 477 observations
█ █ 115.10 ms
█ Validate on cluster hostname=nova-12.lyon.grid5000.fr...
█ █ this cluster contains 476 observations
█ █ 76.94 ms
█ Validate on cluster hostname=paradoxe-32.rennes.grid5000.fr...
█ █ this cluster contains 618 observations
█ █ 140.79 ms
█ Validate on cluster hostname=suzanne...
█ █ this cluster contains 467 observations
█ █ 115.28 ms
█ Validate on cluster hostname=yeti-2.grenoble.grid5000.fr...
█ █ this cluster contains 433 observations
█ █ 150.86 ms
█ 2 min 10 s
MAPE of log_act_duration_per_frame: 76.74%
RMS  of log_act_durat

## Use the model to perform predictions

The idea is to run a huge number of predictions using the trained model in order to explore a very large range of scenarios.

In [ ]:
solver = Solver(quality_model, lambda **kwd: kwd["log_rate"] + (kwd["psnr"]-35)**2)
tested_combinations, loss = solver.solve(video)